##L10: Gold layer and first findings

###Goal
Build da.gold.daily_zone_summary and run four queries that become the findings in your M1 README.

###Step 1. Build the first gold table
One row per day per pickup zone. The grain is stated in the comment.

In [0]:
CREATE OR REPLACE TABLE da.gold.daily_zone_summary
    COMMENT 'Grain: one row per pickup date and pickup zone'
AS 
SELECT t.pickup_date, t.pu_location_id, z.Borough AS pickup_borough, z.Zone AS pickup_zone,
       COUNT(*) AS trips, 
       SUM(t.total_amount) AS total_revenue,
       ROUND(AVG(t.fare_amount), 2) AS avg_fare, 
       ROUND(AVG(t.trip_distance), 2) AS avg_distance,
       SUM(t.tip_amount) AS total_tips,
       SUM(CASE WHEN t.cbd_congestion_fee > 0 THEN 1 ELSE 0 END) AS trips_with_cbd_fee

FROM da.silver.yellow_trips AS t
LEFT JOIN da.bronze.taxi_zones z ON t.pu_location_id = z.LocationID
GROUP BY ALL
;
SELECT COUNT(*), SUM(trips) FROM da.gold.daily_zone_summary;
SELECT COUNT(*), SUM(trips) FROM da.gold.daily_zone_summary WHERE trips_with_cbd_fee>0 LIMIT 10;

###Step 2. Finding 1: congestion pricing appears in the data
Share of trips each day that carry the new central business district fee. Use the chart button, line chart, x = pickup_date, y = pct_with_fee.

In [0]:
SELECT pickup_date, SUM(trips) AS trips,
       ROUND(100 * SUM(trips_with_cbd_fee) / SUM(trips), 1) AS pct_with_fee
FROM da.gold.daily_zone_summary
GROUP BY pickup_date
ORDER BY pickup_date;

###Step 3. Finding 2: tips by payment type

In [0]:
SELECT payment_type, COUNT(*) AS trips,
       ROUND(AVG(tip_amount), 2) AS avg_tip,
       ROUND(100 * SUM(tip_amount) / NULLIF(SUM(fare_amount), 0), 1) AS tip_pct_of_fare
FROM da.silver.yellow_trips GROUP BY payment_type ORDER BY trips DESC;

Databricks visualization. Run in Databricks to view.

###Step 4. Finding 3: airport trips
Rate code 2 is the flat JFK fare, rate code 3 is Newark.

In [0]:
SELECT rate_code_id, 
    COUNT(*) AS trips, 
    ROUND(AVG(total_amount), 2) AS avg_total, 
    ROUND(AVG(trip_distance), 1) AS avg_miles
    
FROM da.silver.yellow_trips 
WHERE rate_code_id IN (1, 2, 3) 
GROUP BY rate_code_id 
ORDER BY rate_code_id
;

Databricks visualization. Run in Databricks to view.

###Step 5. Finding 4: demand by hour of day

In [0]:
SELECT hour(pickup_ts) AS hour_of_day, 
    COUNT(*) AS trips, 
    ROUND(AVG(fare_amount), 2) AS avg_fare

FROM da.silver.yellow_trips 
GROUP BY 1 
ORDER BY 1
;

Databricks visualization. Run in Databricks to view.